# Course 3: Large Language Models
## Session 7: Reasoning models: GRPO and test-time compute

**Goal:** train a tiny policy on arithmetic with nothing but a right-or-wrong reward, and measure what voting at test time adds.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

### 1. The task and a weak starting policy
Problems are `a + b` with a and b from 0 to 9. The policy reads the two numbers and outputs one of 19 possible answers. First it gets a little supervised training on a few examples, like a pretrained model that is often wrong.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
problems = torch.tensor([(a, b) for a in range(10) for b in range(10)]); answers = problems.sum(1)

class Policy(nn.Module):
    def __init__(self, d=32):
        super().__init__(); self.emb = nn.Embedding(10, d); self.net = nn.Sequential(nn.Linear(2 * d, 64), nn.ReLU(), nn.Linear(64, 19))
    def forward(self, p): return self.net(self.emb(p).flatten(1))

policy = Policy(); opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
seen = torch.randperm(100)[:15]                                        # only 15 worked examples
for _ in range(10):
    loss = F.cross_entropy(policy(problems[seen]), answers[seen]); opt.zero_grad(); loss.backward(); opt.step()
greedy_acc = lambda: (policy(problems).argmax(1) == answers).float().mean().item()
print("accuracy after the small supervised start: %.2f" % greedy_acc())

### 2. GRPO
For each problem, sample a group of answers from the current policy and score each with a checker (1 if correct, else 0). An answer's advantage is its reward minus the group's mean, divided by the group's standard deviation. Raise the probability of answers with positive advantage. No value network is needed, and no labelled answers: only the checker.

In [ ]:
def grpo_step(policy, opt, batch, group=8, clip=0.2, entropy_coef=0.2):
    with torch.no_grad():
        old_logp_all = torch.log_softmax(policy(problems[batch]), -1)
        samples = torch.multinomial(old_logp_all.exp(), group, replacement=True)             # (problems, group)
        reward = (samples == answers[batch][:, None]).float()                                 # the verifiable reward
        adv = (reward - reward.mean(1, keepdim=True)) / (reward.std(1, keepdim=True) + 1e-6)
        old_logp = old_logp_all.gather(1, samples)
    for _ in range(2):                                                                        # a couple of updates on the same samples
        logp_all = torch.log_softmax(policy(problems[batch]), -1); logp = logp_all.gather(1, samples)
        ratio = (logp - old_logp).exp()
        loss = -torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv).mean()          # clipped objective, as in PPO
        loss = loss - entropy_coef * -(logp_all.exp() * logp_all).sum(1).mean()               # entropy bonus: keep exploring
        opt.zero_grad(); loss.backward(); opt.step()
    return reward.mean().item()

opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
history = [greedy_acc()]
for step in range(1000):
    r = grpo_step(policy, opt, torch.randint(100, (32,)))
    if (step + 1) % 250 == 0: history.append(greedy_acc()); print(f"step {step + 1:3d}   mean reward of samples {r:.2f}   greedy accuracy {history[-1]:.2f}")
assert history[-1] > 0.85 and history[-1] > history[0] + 0.5

The policy went from mostly wrong to mostly right without ever being shown a correct answer. When a whole group is wrong, every advantage is zero and nothing is learned from that problem; progress depends on occasionally sampling a correct answer. That is why reasoning models start RL from a model that is already sometimes right, and why this code adds an entropy bonus: without it, the policy becomes certain of its wrong answers on some problems and stops improving (try `entropy_coef=0.0`).

### 3. Test-time compute: sample several answers and vote
Take a half-trained policy and compare one sample with a majority vote over many.

In [ ]:
torch.manual_seed(1)
half = Policy(); o2 = torch.optim.Adam(half.parameters(), lr=3e-3)
for _ in range(10): l = F.cross_entropy(half(problems[seen]), answers[seen]); o2.zero_grad(); l.backward(); o2.step()
for _ in range(150): grpo_step(half, o2, torch.randint(100, (32,)))

@torch.no_grad()
def vote_accuracy(model, k, trials=20):
    probs = torch.softmax(model(problems), -1); accs = []
    for _ in range(trials):
        s = torch.multinomial(probs, k, replacement=True)
        accs.append((torch.mode(s, 1).values == answers).float().mean().item())
    return np.mean(accs)

rows = [(k, vote_accuracy(half, k)) for k in [1, 3, 9, 27, 81]]
for k, a in rows: print(f"samples {k:2d}   accuracy {a:.3f}")
print("greedy (the single most likely answer): %.3f" % (half(problems).argmax(1) == answers).float().mean().item())
assert rows[-1][1] > rows[0][1]

Voting helps when the right answer is the most likely one but not certain. It cannot fix problems where the model's favourite answer is wrong, so the curve flattens at the greedy accuracy. More compute at test time buys reliability, not new knowledge.

### 4. A reward that can be gamed
Suppose the checker is sloppy and also accepts any answer of 9 or more as 'close enough' when the true sum is 9 or more. RL optimizes the checker, not your intention.

In [ ]:
torch.manual_seed(0)
gamed = Policy(); o3 = torch.optim.Adam(gamed.parameters(), lr=3e-3)
def sloppy_step(batch, group=8):
    logp_all = torch.log_softmax(gamed(problems[batch]), -1)
    samples = torch.multinomial(logp_all.exp().detach(), group, replacement=True)
    truth = answers[batch][:, None]
    reward = ((samples == truth) | ((truth >= 9) & (samples >= 9))).float()
    adv = (reward - reward.mean(1, keepdim=True)) / (reward.std(1, keepdim=True) + 1e-6)
    loss = -(adv * logp_all.gather(1, samples)).mean(); o3.zero_grad(); loss.backward(); o3.step()
    return reward.mean().item()
for _ in range(400): r = sloppy_step(torch.randint(100, (32,)))
print("reward under the sloppy checker: %.2f | true accuracy: %.2f" % (r, (gamed(problems).argmax(1) == answers).float().mean().item()))

### 5. GRPO on word problems with a small language model (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU. Libraries such as trl provide a GRPOTrainer; check its current documentation for the exact arguments.
# 1. Data: 2,000 arithmetic word problems in Bangla with a number as the answer (generate them from templates).
# 2. Prompt the model to think step by step and to end with a line 'উত্তর: <number>'.
# 3. Reward function: parse the final number; 1.0 if it equals the answer, else 0.0. Add 0.1 for following the format.
# 4. Train a model of at most 1 billion parameters with groups of 8 samples per problem.
# 5. Report accuracy on 200 held-out problems before and after, and read 20 reasoning traces by hand: is the reasoning real, or did the model find a shortcut?

### TODO
1. Set `entropy_coef=0.0` and give the policy a longer supervised start (60 steps). Where does accuracy stall, and why?
2. Change the group size to 2 and to 32. What happens to speed and to stability?
3. In section 4, find the cheapest change to the checker that removes the exploit.